In [0]:
%pip install langchain langchain-community langchainhub langchain-openai langchain-databricks databricks-langchain
 
dbutils.library.restartPython()

In [0]:
OPENAI_API_KEY=""

In [0]:
import os
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [0]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="gpt-4")

question=input("Enter the question")

response=llm.invoke(question)
print(response.content)

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(model="databricks-gpt-oss-20b")
question=input("Enter a question")
response=llm.invoke(question)
print(response.content)

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(model="databricks-claude-3-7-sonnet")
question=input("Enter a question")
response=llm.invoke(question)
print(response.content)

## PROMPT TEMPLATE

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.prompts import PromptTemplate

llm=ChatDatabricks(model="databricks-claude-3-7-sonnet")

prompt_template=PromptTemplate(
    input_variable=["country"],
    template="""You are an expert in traditional cuisines.
    You provide information about a specific dish from a specific country.
    Answer the question: What is the traditional cuisine of {country}
    """
)

response=llm.invoke(prompt_template.format(country="India"))
print(response.content)


In [0]:
from databricks_langchain import ChatDatabricks
from langchain.prompts import PromptTemplate

llm=ChatDatabricks(model="databricks-claude-3-7-sonnet")

prompt_template=PromptTemplate(
    input_variable=["country"],
    template="""You are an expert in traditional cuisines.
    You provide information about a specific dish from a specific country.
    Avoid giving information about fictional places. If its fictional place just give output as 'I dont know'.
    Answer the question: What is the traditional cuisine of {country}
    """
)

response=llm.invoke(prompt_template.format(country="Wakanda"))
print(response.content)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.prompts import PromptTemplate

llm=ChatDatabricks(model="databricks-claude-3-7-sonnet")

prompt_template=PromptTemplate(
    input_variable=["country","no_of_para","language"],
    template="""You are an expert in traditional cuisines.
    You provide information about a specific dish from a specific country.
    Avoid giving information about fictional places. If its fictional place just give output as 'I dont know'.
    Answer the question: What is the traditional cuisine of {country}.
    Answer in {no_of_para} paragraphs in {language} language.
    """
)

response=llm.invoke(prompt_template.format(country="India",no_of_para=2,language="Kanada"))
print(response.content)

In [0]:
# Task 
Welcome to the {city} travel guide!.
    If you're visiting in {month}, here's what you can do:
    1.Must- Visit attractions.
    2.Local Cuisine
    3.Useful phrases in {language}.
    4.Tips for travelling on a {budget} budget


In [0]:
from langchain.prompts import PromptTemplate
from databricks_langchain import ChatDatabricks


llm = ChatDatabricks(model="databricks-gemini-2-5-flash")

prompt_template = PromptTemplate(
    input_variables=[
        "city",
        "month",
        "language",
        "budget"
    ],
    template=(
        "Welcome to the {city} travel guide!\n"
        "If you're visiting in {month}, here is a detailed guide to help you make the most of your trip:\n"
        "Start by exploring the must-visit attractions that showcase the unique culture and history of {city}. "
        "Indulge in the local cuisine, with recommendations for dishes and restaurants that fit your taste and budget. "
        "Learn a few useful phrases in {language} to enhance your travel experience and connect with locals. "
        "Finally, discover practical tips for traveling on a {budget} budget, including affordable accommodation, transportation, and activities.\n"
        "Please provide the information in a single, well-structured paragraph."
    )
)
print("Travel Guide App")

city=input("enter your travel city")
month=input("enter your travel month")
language=input("enter your language")
budget=input("enter your budget")


if city:
    response=llm.invoke(prompt_template.format(city=city,month=month,language=language,budget=budget))
    print(response.content)

https://docs.databricks.com/aws/en/machine-learning/model-serving/query-chat-models?language=Databricks%C2%A0Python%C2%A0SDK#-query-examples

In [0]:
from langchain_core.messages import HumanMessage, SystemMessage
from databricks_langchain import ChatDatabricks

messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage(content="What is a mixture of experts model?"),
]

llm = ChatDatabricks(model="databricks-claude-sonnet-4-5")
llm.invoke(messages)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
response = w.serving_endpoints.query(
    name="databricks-claude-sonnet-4-5",
    messages=[
        ChatMessage(
            role=ChatMessageRole.SYSTEM, content="You are a helpful assistant."
        ),
        ChatMessage(
            role=ChatMessageRole.USER, content="What is a mixture of experts model?"
        ),
    ],
    max_tokens=128,
)
print(f"RESPONSE:\n{response.choices[0].message.content}")

In [0]:
%sql
SELECT ai_query(
    "databricks-claude-sonnet-4-5",
    "Can you explain AI in ten words?"
  )

In [0]:

import mlflow.deployments

# Only required when running this example outside of a Databricks Notebook
#export DATABRICKS_HOST="https://<workspace_host>.databricks.com"
#export DATABRICKS_TOKEN="dapi-your-databricks-token"

client = mlflow.deployments.get_deploy_client("databricks")

chat_response = client.predict(
    endpoint="databricks-claude-sonnet-4-5",
    inputs={
        "messages": [
            {
              "role": "user",
              "content": "Hello!"
            },
            {
              "role": "assistant",
              "content": "Hello! How can I assist you today?"
            },
            {
              "role": "user",
              "content": "What is a mixture of experts model??"
            }
        ],
        "temperature": 0.1,
        "max_tokens": 20
    }
)


print(chat_response)

In [0]:
import mlflow

# Enable automatic tracing for LangChain — all subsequent LangChain calls
# will be traced and logged to the active MLflow experiment automatically.
mlflow.langchain.autolog()

# Now query a model via LangChain — the trace is captured automatically
from databricks_langchain import ChatDatabricks
from langchain.prompts import PromptTemplate

llm = ChatDatabricks(model="databricks-claude-sonnet-4-5")

prompt_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in 3 sentences."
)

chain = prompt_template | llm

response = chain.invoke({"topic": "mixture of experts models"})
print(response.content)

In [0]:
%pip install mlflow>=3.0 --upgrade

In [0]:
import mlflow

# Enable automatic tracing for LangChain — all subsequent LangChain calls
# will be traced and logged to the active MLflow experiment automatically.
mlflow.langchain.autolog()



from databricks_langchain import ChatDatabricks
from langchain.prompts import PromptTemplate

llm1=ChatDatabricks(model="databricks-gpt-oss-20b")
llm2=ChatDatabricks(model="databricks-claude-opus-4-1")
title_prompt=PromptTemplate(
    input_variable=["topic"],
    template="""You are an experienced speech writer.
    You need to craft an impactful title for a speech 
    on the following topic: {topic}
    Answer exactly with one title.	
    """)

speech_prompt=PromptTemplate(
    input_variable=["title"],
    template="""You need to write a powerful speech of 250 words 
    for the following title:{title}
    """)

first_chain= title_prompt| llm1 # generate title 
second_chain=speech_prompt | llm2 # generate speech
final_chain=first_chain | second_chain

topic=input("Enter the topic name: ")

#chain=first_chain | second_chain

if topic:
    response=final_chain.invoke({"topic":topic})
    print(response.content)


In [0]:
%sql
Use 2 different LLMS. 

Blog Post Generator:
  You are a professional blogger.
    Create an outline for a blog post on the following topic: {topic}
    The outline should include:
    - Introduction
    - 3 main points with subpoints
    - Conclusion


  You are a professional blogger.
    Write an engaging introduction paragraph based on the following
    outline:{outline}
    The introduction should hook the reader and provide a brief
    overview of the topic